In [221]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
import random
%matplotlib inline

In [222]:
words = open('../names.txt').read().splitlines()

In [223]:
stoi = {c:i+1 for i,c in enumerate(sorted(set(''.join(words))))}
stoi['.'] = 0
itos = {i:c for c, i in stoi.items()}

In [224]:
def build_dataset(data):
    block_size = 3
    X = []
    Y = []
    for word in data:
        start = [0] * (block_size-1)
        paddedWord = '.' + word + '.'
        subL = []
        for c in paddedWord:
            subL.append(stoi[c])
        for x, y in zip(subL, subL[1:]):
            start.append(x)
            X.append(start)
            Y.append(y)
            start = start[1:]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y


In [225]:
random.seed(42)
random.shuffle(words)
eightyP = int(.8 * len(words))
ninetyP = int(.9 * len(words))
Xtrain, Ytrain = build_dataset(words[:eightyP])
Xdev, Ydev = build_dataset(words[eightyP:ninetyP])
Xtest, Ytest = build_dataset(words[ninetyP:])

In [245]:
g = torch.Generator().manual_seed(42)
C = torch.randn(27, 10, generator=g, requires_grad=True)
W1 = torch.randn(30, 200, generator=g, requires_grad=True)
b1 = torch.randn(200, generator=g, requires_grad=True)
W2 = torch.randn(200, 27, generator=g, requires_grad=True)
b2 = torch.randn(27, generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

In [227]:
def zeroGrad():
    for val in parameters:
        val.grad = None

In [ ]:
def weightUpdate(i):
    lr = 0.1 if i<=90 else 0.01
    for val in parameters:
        val.data -= lr * val.grad

In [236]:
def trainingLoop(Xs, Ys):
    for i in range(100):
        emb = torch.flatten(C[Xs], start_dim=1)
        h = torch.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Ys)
        zeroGrad()
        loss.backward()
        weightUpdate(i)
    return loss.item()

        

In [266]:
trainingLoop(Xtrain, Ytrain)

2.958645820617676

In [267]:
trainingLoop(Xdev, Ydev)

2.6698646545410156

In [268]:
trainingLoop(Xtest, Ytest)

2.8173251152038574

In [274]:
g = torch.Generator().manual_seed(2147483647)
block_size = 3
for _ in range(20):

    out = []
    context = [0] * block_size  # initialize with all ...
    while True:
        x = torch.tensor([context])
        emb = torch.flatten(C[x], start_dim=1)
        #emb = C[torch.tensor([context])]  # (1, block_size, d)
        h = torch.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(''.join(itos[i] for i in out))

cer.
maleah.
makile.
kayda.
malimitton.
ramays.
kat.
arreen.
peby.
ber.
goto.
michaselnal.
ker.
mdauma.
emite.
adsheniav.
jayreell.
mhina.
ghson.
las.
